In [1]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

In [4]:
# Directories 
ROOT_DIR = Path("/Users/bmacedo/Desktop/final_WM")
HARMONIZED_DIR = ROOT_DIR / "output_data" / "harmonized"
OUTPUT_DIR = ROOT_DIR / "output_data" / "results" 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load harmonized data

# ---- MAIN DATA ----
df_harmonized_exposome_A = pd.read_csv(HARMONIZED_DIR / "df_harmonized_exposome_A_12_10.csv")
df_harmonized_exposome_B = pd.read_csv(HARMONIZED_DIR / "df_harmonized_exposome_B_12_10.csv")

# ---- COGNITION ----
df_harmonized_exposome_A_cog = pd.read_csv(HARMONIZED_DIR / "df_harmonized_exposome_cognition_A_12_10.csv")
df_harmonized_exposome_B_cog = pd.read_csv(HARMONIZED_DIR / "df_harmonized_exposome_cognition_B_12_10.csv")

# ---- EDUCATION/INCOME SENSITIVITY ----
df_harmonized_exposome_A_sens = pd.read_csv(HARMONIZED_DIR / "df_income_sens_A_11_30.csv")
df_harmonized_exposome_B_sens = pd.read_csv(HARMONIZED_DIR / "df_income_sens_B_11_30.csv")

# ---- Logging ----
print("[MAIN] A:", df_harmonized_exposome_A.shape)
print("[MAIN] B:", df_harmonized_exposome_B.shape)
print("[COG] A:", df_harmonized_exposome_A_cog.shape)
print("[COG] B:", df_harmonized_exposome_B_cog.shape)
print("[SENS] A:", df_harmonized_exposome_A_sens.shape)
print("[SENS] B:", df_harmonized_exposome_B_sens.shape)

[MAIN] A: (8428, 3677)
[MAIN] B: (8428, 3677)
[COG] A: (7877, 3681)
[COG] B: (7877, 3681)
[SENS] A: (6809, 3672)
[SENS] B: (6809, 3672)


In [5]:
MODEL_PREFIXES = {"DKI": "DKI_", "NODDI": "NODDI_", "MAPMRI": "MAPMRI_", "GQI": "GQI_"}

def split_msmt(col):
    # Parse 'msmt_{BUNDLE}_{METRIC}' -> (bundle, metric), returns (None, None) if not msmt_*
    if not col.startswith("msmt_"):
        return None, None
    rest = col[len("msmt_"):] # removes msmt_ prefix... all that is left is bundle_metric
    bundle, metric = rest.split("_", 1)  # only the first '_' splits bundle, metric
    return bundle, metric

def model_of_metric(metric):
    # Returns one of {'DKI','NODDI','MAPMRI','GQI','macro'}
    for model, pref in MODEL_PREFIXES.items():
        if metric.startswith(pref): return model
    return "macro"  # msmt_* but no micro prefix -> macrostructure

def column_kind(col):
    # Classify a column: 'covariate' if not msmt_*, otherwise one of {'DKI','NODDI','MAPMRI','GQI','macro'}
    bundle, metric = split_msmt(col)
    if bundle is None: return "covariate"
    return model_of_metric(metric)

def catalog_models(df):
    return pd.Series({c: column_kind(c) for c in df.columns}, name="kind")

def filter_df_by_models(df, include_models=("DKI",), keep_macro=True, keep_covariates=True):
    # keeps all covariates if keep_covariates, all macrostructure metrics if keep_macro, micro metrics only from include_models
    keep_cols = []
    for c in df.columns:
        kind = column_kind(c)
        if kind == "covariate" and keep_covariates: keep_cols.append(c)
        elif kind == "macro" and keep_macro: keep_cols.append(c)
        elif kind in include_models: keep_cols.append(c)
    return df.loc[:, keep_cols]

In [13]:
def save_filtered(df, *, include_models, keep_macro, keep_covariates, fname):
    filtered_df = filter_df_by_models(df, include_models=include_models, keep_macro=keep_macro, keep_covariates=keep_covariates)
    filtered_df.to_csv(OUTPUT_DIR / fname, index=False)
    print(f"[SAVE] {fname}: {filtered_df.shape}")

# label, include_models, keep_macro
specs = [("macro", (), True), ("NODDI", ("NODDI",), False), ("macro_plus_NODDI", ("NODDI",), True), 
         ("DKI", ("DKI",), False), ("macro_plus_DKI", ("DKI",), True)]

for label, models, keep_macro in specs:
    # ---- MAIN ----
    save_filtered(df_harmonized_exposome_A, include_models=models, keep_macro=keep_macro, keep_covariates=True, 
                  fname=f"dfA_{label}_12_10.csv")
    save_filtered(df_harmonized_exposome_B, include_models=models, keep_macro=keep_macro, keep_covariates=True, 
              fname=f"dfB_{label}_12_10.csv")

    # ---- COGNITION ----
    save_filtered(df_harmonized_exposome_A_cog, include_models=models, keep_macro=keep_macro, keep_covariates=True,
                  fname=f"dfA_{label}_cog_12_10.csv")
    save_filtered(df_harmonized_exposome_B_cog, include_models=models, keep_macro=keep_macro, keep_covariates=True,
                  fname=f"dfB_{label}_cog_12_10.csv")

    # ---- SENSITIVITY ----
    save_filtered(df_harmonized_exposome_A_sens, include_models=models, keep_macro=keep_macro, keep_covariates=True, 
                  fname=f"dfA_{label}_sens_11_30.csv")
    save_filtered(df_harmonized_exposome_B_sens, include_models=models, keep_macro=keep_macro, keep_covariates=True, 
                  fname=f"dfB_{label}_sens_11_30.csv")

[SAVE] dfA_macro_12_10.csv: (8428, 1135)
[SAVE] dfB_macro_12_10.csv: (8428, 1135)
[SAVE] dfA_macro_cog_12_10.csv: (7877, 1139)
[SAVE] dfB_macro_cog_12_10.csv: (7877, 1139)
[SAVE] dfA_macro_sens_11_30.csv: (6809, 1130)
[SAVE] dfB_macro_sens_11_30.csv: (6809, 1130)
[SAVE] dfA_NODDI_12_10.csv: (8428, 205)
[SAVE] dfB_NODDI_12_10.csv: (8428, 205)
[SAVE] dfA_NODDI_cog_12_10.csv: (7877, 209)
[SAVE] dfB_NODDI_cog_12_10.csv: (7877, 209)
[SAVE] dfA_NODDI_sens_11_30.csv: (6809, 200)
[SAVE] dfB_NODDI_sens_11_30.csv: (6809, 200)
[SAVE] dfA_macro_plus_NODDI_12_10.csv: (8428, 1321)
[SAVE] dfB_macro_plus_NODDI_12_10.csv: (8428, 1321)
[SAVE] dfA_macro_plus_NODDI_cog_12_10.csv: (7877, 1325)
[SAVE] dfB_macro_plus_NODDI_cog_12_10.csv: (7877, 1325)
[SAVE] dfA_macro_plus_NODDI_sens_11_30.csv: (6809, 1316)
[SAVE] dfB_macro_plus_NODDI_sens_11_30.csv: (6809, 1316)
[SAVE] dfA_DKI_12_10.csv: (8428, 577)
[SAVE] dfB_DKI_12_10.csv: (8428, 577)
[SAVE] dfA_DKI_cog_12_10.csv: (7877, 581)
[SAVE] dfB_DKI_cog_12_10.csv: 